# Lezione 11 — Dentro `fit`: batch, epoche e autodiff

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezioni 8-10.

**Cosa saprai fare alla fine:** aprire la scatola di `model.fit` — sapere
cosa sono batch ed epoche, cosa fa l'autodiff, e saper scrivere un training
loop a mano in TensorFlow. Da qui in poi nessun training sarà più una
scatola nera.

## Parte 1 — Teoria: batch ed epoche

Nella Lezione 9 il gradiente lo calcolavi su **tutto** il train a ogni
passo. Con 213 righe va benissimo; con 10 milioni no. La soluzione
standard: il **mini-batch**.

- un **batch** è un sottoinsieme (tipicamente 32) di esempi: il gradiente
  calcolato sul batch è una *stima rumorosa* del gradiente vero — ma costa
  una frazione, e il rumore aiuta perfino a non incastrarsi;
- un'**epoca** è un giro completo su tutto il train, un batch alla volta;
- `fit(epochs=150, batch_size=32)` = 150 giri, ~7 aggiornamenti a giro.

Il ciclo che conosci diventa: per ogni batch → forward → loss → gradiente →
passo. Identico alla Lezione 9, solo su fette di dati.

## Parte 2 — Teoria: l'autodiff

Nella Lezione 8 i gradienti li hai scritti a mano (per due parametri) e
verificati numericamente. Per una rete con migliaia di parametri serve di
meglio: la **differenziazione automatica**.

L'idea: TensorFlow **registra** ogni operazione che esegui (una "tape",
un nastro) e poi applica la chain rule **all'indietro** lungo il nastro —
la backpropagation, meccanizzata. Non è una derivata numerica (lenta,
approssimata) né simbolica (esplosiva): è la chain rule esatta applicata al
grafo delle operazioni realmente eseguite.

Verifichiamolo sulla funzione della Lezione 8, di cui conosci la derivata
esatta (2x + 3):

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # silenzia i log informativi di TF

import pandas as pd
import keras

keras.utils.set_random_seed(42)            # riproducibilita' (Lezione 3!)
print('Keras', keras.__version__)

In [ ]:
import tensorflow as tf

x = tf.Variable(2.0)
with tf.GradientTape() as nastro:
    y = x**2 + 3 * x           # TF registra le operazioni
gradiente = nastro.gradient(y, x)
print(f'autodiff: {gradiente.numpy():.4f}   derivata esatta 2x+3 in x=2: 7.0000')

Stesso numero della verifica numerica della Lezione 8 — ma esatto e su
qualsiasi funzione, comprese le reti a migliaia di parametri. `fit` usa
esattamente questo meccanismo a ogni batch.

## Parte 3 — Esercizio guidato

La Lezione 8 minimizzava `g(x) = (x-4)² + 1` con derivata scritta a mano.
Il tuo compito: rifallo con l'autodiff — `tf.Variable` che parte da -2.0,
20 passi, learning rate 0.2, gradiente dal `GradientTape` (dentro il ciclo)
— e verifica di arrivare vicino a 4.

**Prova tu:**

In [ ]:
# Scrivi qui: variabile, ciclo di 20 passi con GradientTape,
# aggiornamento x.assign_sub(passo * gradiente)

pass

### Soluzione spiegata

- il `GradientTape` va **dentro** il ciclo: registra le operazioni di quel
  passo (il nastro si consuma a ogni `gradient`);
- `assign_sub` è il `x = x - passo * grad` della Lezione 8, nella forma che
  aggiorna la variabile TF sul posto;
- il risultato coincide con la versione a mano: l'autodiff non cambia la
  matematica, la automatizza.

In [ ]:
x = tf.Variable(-2.0)
for _ in range(20):
    with tf.GradientTape() as nastro:
        g = (x - 4.0)**2 + 1.0
    x.assign_sub(0.2 * nastro.gradient(g, x))
print(f'dopo 20 passi: x = {x.numpy():.4f}  (minimo teorico: 4)')

## Parte 4 — Il progetto: Memory AI Lab, passo 11

Riscriviamo il training della rete della Lezione 10 **senza** `fit`: stesso
modello, stesso risultato, ma ogni pezzo del ciclo visibile. Questo è il
"model fit under the hood" — e da qui in avanti, quando un training si
comporterà in modo strano, saprai entrarci dentro.

In [ ]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_features_{n}.csv') for n in ['train', 'val']}
X = {n: f.drop(columns='target').to_numpy().astype('float32') for n, f in insiemi.items()}
y = {n: f['target'].to_numpy() for n, f in insiemi.items()}

keras.utils.set_random_seed(42)
modello = keras.Sequential([
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(4, activation='softmax'),
])
loss_fn = keras.losses.SparseCategoricalCrossentropy()
ottimizzatore = keras.optimizers.Adam()

dataset = tf.data.Dataset.from_tensor_slices((X['train'], y['train'])).batch(32)

for epoca in range(60):
    for X_batch, y_batch in dataset:                       # i batch della Parte 1
        with tf.GradientTape() as nastro:                  # l'autodiff della Parte 2
            probabilita = modello(X_batch, training=True)  # forward (Lezione 7)
            loss = loss_fn(y_batch, probabilita)           # contratto (Lezione 9)
        gradienti = nastro.gradient(loss, modello.trainable_variables)
        ottimizzatore.apply_gradients(zip(gradienti, modello.trainable_variables))
    if epoca % 20 == 0:
        print(f'epoca {epoca:2d}: loss sul batch finale {float(loss):.3f}')

predizioni = modello(X['val']).numpy().argmax(axis=1)
print(f"\naccuratezza validation col loop scritto a mano: {(predizioni == y['val']).mean():.0%}")

Confronta questo loop col ciclo NumPy della Lezione 9: la struttura è
**identica** — forward, loss, gradiente, passo — con tre upgrade: i batch,
l'autodiff al posto del gradiente scritto a mano, e Adam al posto del passo
fisso. `fit` non fa nient'altro (più i callback, che vedrai nella
Lezione 12).

## Cosa hai imparato

- **Batch**: gradiente stimato su una fetta di dati — più economico,
  rumoroso il giusto. **Epoca**: un giro completo.
- L'**autodiff** registra le operazioni e applica la chain rule
  all'indietro: la backpropagation meccanizzata, esatta.
- Il training loop di Keras è il tuo ciclo della Lezione 9 con batch +
  autodiff + optimizer: l'hai riscritto e dà lo stesso risultato.

## Quiz

1. Con 213 esempi e `batch_size=32`, quanti aggiornamenti dei pesi avvengono
   in un'epoca?
2. Perché il gradiente calcolato su un batch è "rumoroso", e perché non è
   un problema?
3. Che differenza c'è tra la derivata del `GradientTape` e la
   `derivata_numerica` della Lezione 8?

<details>
<summary><b>Apri le risposte</b></summary>

1. ceil(213/32) = 7 aggiornamenti per epoca.
2. È calcolato su un campione, quindi è una stima della media vera. Non è
   un problema perché in media punta nella direzione giusta, costa poco, e
   il rumore aiuta a uscire da punti piatti o minimi mediocri.
3. Il tape applica la chain rule esatta alle operazioni registrate (errore
   solo di virgola mobile); la derivata numerica perturba l'input e misura,
   quindi è approssimata e costa una valutazione per parametro — utile per
   verificare, impraticabile per addestrare.
</details>

## Fonti

- TensorFlow, *Introduction to gradients and automatic differentiation*:
  https://www.tensorflow.org/guide/autodiff
- TensorFlow, *Writing a training loop from scratch* (Keras):
  https://keras.io/guides/writing_a_custom_training_loop_in_tensorflow/

## Prossima lezione

Il training funziona — anche troppo: una rete abbastanza grande può
imparare il train **a memoria**. Overfitting, curve di apprendimento,
dropout ed early stopping: la Lezione 12 chiude il primo blocco della
Fase 2, e il progetto usa — per la prima e unica volta — il suo test set.